# Runtimes, Event Hooks, and State Debugging

This notebook demonstrates three advanced capabilities:
- Comparing runtime behavior (`SimpleRuntime` vs `SoftRZRuntime`)
- Collecting metrics and circuits via event hooks
- Capturing simulator state outputs for debugging

In [ ]:
from guppylang.decorator import guppy
from guppylang.std.quantum import qubit, h, cx, measure
from guppylang.std.builtins import result
from guppylang.std.debug import state_result

from selene_sim.build import build
from selene_sim import Quest, Stim, SimpleRuntime, SoftRZRuntime
from selene_sim.event_hooks import MetricStore, CircuitExtractor, MeasurementExtractor, MultiEventHook

In [ ]:
@guppy
def ghz_with_measurement() -> None:
    q0 = qubit()
    q1 = qubit()
    q2 = qubit()
    h(q0)
    cx(q0, q1)
    cx(q1, q2)
    result("c0", measure(q0))
    result("c1", measure(q1))
    result("c2", measure(q2))

runner = build(ghz_with_measurement.compile(), "runtime_hook_demo")

## 1) Compare runtimes

Observable user results should match, while post-runtime gate counts may differ.

In [ ]:
simulator = Quest(random_seed=2025)

simple_metrics = MetricStore()
simple_out = dict(
    runner.run(
        simulator=simulator,
        runtime=SimpleRuntime(),
        n_qubits=3,
        event_hook=simple_metrics,
    )
)

soft_metrics = MetricStore()
soft_out = dict(
    runner.run(
        simulator=simulator,
        runtime=SoftRZRuntime(),
        n_qubits=3,
        event_hook=soft_metrics,
    )
)

print("Outputs equal:", simple_out == soft_out)
print("Simple post-runtime metrics:", simple_metrics.shots[0]["post_runtime"])
print("SoftRZ post-runtime metrics:", soft_metrics.shots[0]["post_runtime"])

## 2) Collect circuits, measurements, and metrics in one run

In [ ]:
metrics = MetricStore()
circuits = CircuitExtractor()
measurements = MeasurementExtractor()
hook = MultiEventHook([metrics, circuits, measurements])

shots = [dict(shot) for shot in runner.run_shots(
    simulator=Stim(random_seed=99),
    n_qubits=3,
    n_shots=3,
    event_hook=hook,
)]

print("shots:", shots)
print("metric entries:", len(metrics.shots))
print("circuit entries:", len(circuits.shots))
print("measurement entries:", len(measurements.shots))

## 3) State debugging with `state_result`

`state_result` emits a named state artifact that can be parsed with simulator helpers.

In [ ]:
@guppy
def state_demo() -> None:
    q0 = qubit()
    q1 = qubit()
    h(q0)
    cx(q0, q1)
    state_result("bell_state", q0, q1)

state_runner = build(state_demo.compile(), "state_demo")
state_shot = list(state_runner.run(Quest(), n_qubits=2))
state_map = Quest.extract_states_dict(state_shot)

print("state labels:", list(state_map.keys()))
print("state vector:", state_map["bell_state"].get_single_state())